# Create Dataset

This notebook creates a dataset containing training, testing and validation splits with options for choosing a desired amount of X % of each month of the year (provided such yearly dataset) and with X % upturning sampling of flights for better flight turning prediction.

In [1]:
import sys
from pathlib import Path
import importlib

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import from utils package
import utils
importlib.reload(utils.utils)
importlib.reload(utils)

from utils import (
    WindowParams,
    SplitConfig,
    SamplingConfig,
    StatsConfig,
    TurnSampling,
    VerticalSampling,
    build_or_load_dataset,
    collect_parquet_files,
    load_data_from_files,
    filter_and_check,
    summarize_motion_distribution,
)

print("Imports successful!")

# Debug: ensure we import the *local* utils from this repo
import inspect
print('utils module path:', getattr(utils, '__file__', None))
print('utils.utils module path:', getattr(utils.utils, '__file__', None))
print('build_or_load_dataset signature:', inspect.signature(build_or_load_dataset))


/home/fusg/VT_2/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful!


In [2]:
# Configuration
DATA_DIR = "/store/Projects_CRM/CRM4_data/trajectory/new_approach/preprocessed/2024"
DATA_FRAC = 0.20  # 20% of data from all months
TURN_UPSAMPLING = 0.15  # 15% turn upsampling
MIN_FL = 195 # Minimum flight level accepted (only en-route trajectories)

# Window parameters
wparams = WindowParams(
    input_len=60,      # 60 historical time steps (1 minute at 1Hz)
    output_horizon=60,  # 60 future time steps (1 minute at 5Hz)
    output_stride=5,    # 5 second intervals
    overlap=False       # No overlap between windows
)

# Split configuration
scfg = SplitConfig(
    train_frac=0.7,   # 70% training
    val_frac=0.15,    # 15% validation
    split_seed=42     # Reproducible splits
)

# Sampling configuration with 15% turn upsampling
samp = SamplingConfig(
    n_train=2_000_000,  # 2M training samples
    n_val=500_000,      # 500K validation samples
    n_test=400_000,     # 400K test samples
    train_turn=TurnSampling(
        min_turn_frac=TURN_UPSAMPLING,  # 15% turn upsampling
        turn_thr=0.01,
        consec=3,
        consider_hist=True,
        consider_future=True
    ),
    val_turn=TurnSampling(
        min_turn_frac=0.0,
        turn_thr=0.01,
        consec=3,
        consider_hist=True,
        consider_future=True
    ),
    test_turn=TurnSampling(
        min_turn_frac=0.0,  # No turn upsampling for test
        turn_thr=0.01,
        consec=3,
        consider_hist=True,
        consider_future=True
    ),
    train_vertical=VerticalSampling(
        min_level_frac=0.50,
        min_climb_frac=0.20,
        min_descend_frac=0.20,
        vz_thr=0.5,
        consec=3,
        consider_hist=True,
        consider_future=True,
    ),
    val_vertical=VerticalSampling( 
        min_level_frac=0.0,
        min_climb_frac=0.0,
        min_descend_frac=0.0,
        vz_thr=0.5,
        consec=3,
        consider_hist=True,
        consider_future=True,
    ),
    test_vertical=VerticalSampling( # No upsampling for test
        min_level_frac=0.0, 
        min_climb_frac=0.0,
        min_descend_frac=0.0,
        vz_thr=0.5,
        consec=3,
        consider_hist=True,
        consider_future=True,
    ),
)

# Normalization statistics configuration
stats_cfg = StatsConfig(
    stats_seed=1234,
    stats_sample_size=2_000_000  # Sample size for computing normalization stats
)

print("Configuration set up!")
print(f"  Data fraction: {DATA_FRAC*100}%")
print(f"  Turn upsampling: {TURN_UPSAMPLING*100}%")
print(f"  Min FL: {MIN_FL}")

Configuration set up!
  Data fraction: 20.0%
  Turn upsampling: 15.0%
  Min FL: 195


In [3]:
# Create dataset
print("Creating dataset...")
print("=" * 70)

# Step 1: Collect parquet files (20% from each month)
print("\nStep 1: Collecting parquet files...")
parquet_files = collect_parquet_files(DATA_DIR, DATA_FRAC)

# Step 2: Load and combine data
print("\nStep 2: Loading and engineering data...")
df = load_data_from_files(parquet_files)

Creating dataset...

Step 1: Collecting parquet files...

Step 2: Loading and engineering data...


In [20]:
# Step 3 : Filter and check data
df = filter_and_check(df, MIN_FL, mode="segments")

[filter_and_check] summary
  total flights before FL filter: 288512
  flights kept (min FL >= 195): 153208
  flights dropped by FL: 135304
  total rows before FL filter: 306179783
  total rows after FL filter: 132720694
  NaN rows in retained data: 185578
  flights with NaN in retained data: 245
  NaN counts by column: {'timestamp': 0, 'x': 0, 'y': 0, 'z': 0, 'vx': 184754, 'vy': 184754, 'vz': 183832}
  1s cadence failures in retained data: 345829 rows
  flights with 1s spacing issues: 116909


In [29]:
# Step 4: Computes summary statistics
summary = summarize_motion_distribution(
    df,
    wparams,
    turn=utils.TurnSampling(
        min_turn_frac=TURN_UPSAMPLING,
        turn_thr=0.01,
        consec=3,
        consider_hist=True,
        consider_future=True,
    ),
    vertical=utils.VerticalSampling(
        min_level_frac=0.0,
        min_climb_frac=0.0,
        min_descend_frac=0.0,
        vz_thr=0.5,
        consec=3,
        consider_hist=True,
        consider_future=True,
    ),
)

[summarize_motion_distribution]
  total valid windows: 89255587
  turn windows: 1952805 (2.19%)
  vertical window counts:
    level: 50070715
    climb: 14228913
    descent: 13738208
    mixed: 11217751
  joint counts (vertical x turn):
    level: turn=689649, no_turn=49381066
    climb: turn=535908, no_turn=13693005
    descent: turn=331794, no_turn=13406414
    mixed: turn=395454, no_turn=10822297


In [4]:
# Step 5: Build or load dataset
print("\nStep 3: Building dataset...")
(X_train, Y_train, C_train,
 X_val, Y_val, C_val,
 X_test, Y_test, C_test,
 norm_stats, meta_train, meta_val, meta_test,
 manifest, summary) = build_or_load_dataset(
    df=df,
    wparams=wparams,
    scfg=scfg,
    samp=samp,
    stats_cfg=stats_cfg,
    min_fl=MIN_FL,
    min_fl_mode="segments",
)

print("=" * 70)
print("Dataset creation complete!")
print(f"  Train samples: {len(X_train)}")
print(f"  Val samples: {len(X_val)}")
print(f"  Test samples: {len(X_test)}")


Step 3: Building dataset...
[cache] miss: c3155eaa7dc6786d -> building…
Dataset creation complete!
  Train samples: 2000000
  Val samples: 500000
  Test samples: 400000
